## Bronze Layer — Activity Weather
Extracts weather conditions for outdoor activities from Garmin API.

One row per activity with weather data (indoor activities have no weather).
Filters for activities from 2021-2026.

In [0]:
%pip install garminconnect

In [0]:
import pandas as pd
from garminconnect import Garmin
import json
import time

In [0]:
email = dbutils.secrets.get(scope="garmin", key="email")
password = dbutils.secrets.get(scope="garmin", key="password")
print("Credentials loaded ✅")

In [0]:
api = Garmin(email=email, password=password)
api.login()
print("Garmin API connected ✅")

In [0]:
# Get activities from 2021-2026
df_activities = spark.table("garmin_lakehouse.raw.activities") \
    .select("activityId", "startTimeLocal") \
    .toPandas()

df_activities['year'] = pd.to_datetime(df_activities['startTimeLocal']).dt.year
df_filtered = df_activities[df_activities['year'].isin([2021, 2022, 2023, 2024, 2025, 2026])]

activity_ids = df_filtered['activityId'].tolist()
print(f"Found {len(activity_ids)} activities (2021-2026)")

In [0]:
# Extract weather for each activity
weather_results = []
total = len(activity_ids)

for idx, activity_id in enumerate(activity_ids, 1):
    try:
        weather = api.get_activity_weather(activity_id)
        
        if weather:
            if isinstance(weather, dict):
                weather['activityId'] = activity_id
                weather_results.append(weather)
            elif isinstance(weather, list):
                for w in weather:
                    w['activityId'] = activity_id
                    weather_results.append(w)
    except:
        pass  # Activity has no weather (indoor or no GPS)
    
    # Progress update every 100 activities
    if idx % 100 == 0 or idx == total:
        print(f"  Processed {idx}/{total} activities... ({len(weather_results)} with weather)")
    
    time.sleep(0.1)  # Rate limiting

print(f"\n✅ Found weather data for {len(weather_results)} activities")

In [0]:
if not weather_results:
    print("⚠️ No weather data found - skipping table creation")
else:
    # Convert to DataFrame
    df_weather = pd.DataFrame(weather_results)
    
    # Drop windGust column to avoid void type
    if 'windGust' in df_weather.columns:
        df_weather = df_weather.drop(columns=['windGust'])
    
    # Serialize complex fields to JSON
    for col in df_weather.columns:
        if df_weather[col].apply(lambda x: isinstance(x, (dict, list))).any():
            df_weather[col] = df_weather[col].apply(
                lambda x: json.dumps(x) if isinstance(x, (dict, list)) else x
            )
    
    print(f"Columns: {len(df_weather.columns)}, Rows: {len(df_weather)}")

In [0]:
if weather_results:
    # Write to Delta table
    spark.createDataFrame(df_weather) \
        .write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("garmin_lakehouse.raw.activity_weather")
    
    row_count = spark.table("garmin_lakehouse.raw.activity_weather").count()
    print(f"✅ activity_weather written: {row_count} rows")
else:
    print("⚠️ No data to write")

In [0]:
if spark.catalog.tableExists("garmin_lakehouse.raw.activity_weather"):
    df_sample = spark.table("garmin_lakehouse.raw.activity_weather").select(
        "issueDate", "temp", "apparentTemp", "dewPoint", "relativeHumidity",
        "windDirection", "windDirectionCompassPoint", "windSpeed",
        "latitude", "longitude", "weatherStationDTO", "weatherTypeDTO", "activityId"
    )
    print(f"Total rows: {df_sample.count()}")
    display(df_sample.limit(20))
else:
    print("Table not created - no weather data available")